# DX 704 Week 1 Project

This week's project will build a portfolio risk and return model, and make investing recommendations for hypothetical clients.
You will collect historical data, estimate returns and risks, construct efficient frontier portfolios, and sanity check the certainty of the maximum return portfolio.

The full project description and a template notebook are available on GitHub at the following link.

https://github.com/bu-cds-dx704/dx704-project-01


Feel free to use optimization tools or libraries (such as CVXOPT or scipy.optimize) to perform any calculations required for this mini project.

### Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Collect Data

Collect historical monthly price data for the last 24 months covering 6 different stocks.
The data should cover 24 consecutive months including the last month that ended before this week's material was released on Blackboard.
To be clear, if a month ends between the Blackboard release and submitting your project, you do not need to add that month.

The six different stocks must include AAPL, SPY and TSLA.
At least one of the remaining 3 tickers must start with the same letter as your last name (e.g. professor Considine could use COIN).
This is to encourage diversity in what stocks you analyze; if you discuss this project with classmates, please make sure that you pick different tickers to differentiate your work.
Do not pick stocks with fewer than 24 consecutive months of price data.

In [4]:
# YOUR CHANGES HERE
import pandas as pd
import yfinance as yf

tickers = ["AAPL", "SPY", "TSLA", "BAC", "NVDA", "GOOG"]

historical_prices = None

for t in tickers:
    ticker = yf.Ticker(t)
    history = ticker.history(start="2024-09-01", end="2026-09-01")

    monthly_prices = history.groupby(history.index.to_period("M")).tail(1)["Close"]

    if historical_prices is None:
        historical_prices = monthly_prices
    else:
        historical_prices = pd.concat([historical_prices, monthly_prices], axis=1)

historical_prices.columns = tickers
historical_prices.index = pd.to_datetime(historical_prices.index).strftime("%Y-%m-%d")
historical_prices.index.name = "date"

historical_prices

/tmp/ipykernel_19096/1095469184.py:13: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  monthly_prices = history.groupby(history.index.to_period("M")).tail(1)["Close"]
/tmp/ipykernel_19096/1095469184.py:13: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  monthly_prices = history.groupby(history.index.to_period("M")).tail(1)["Close"]
/tmp/ipykernel_19096/1095469184.py:13: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  monthly_prices = history.groupby(history.index.to_period("M")).tail(1)["Close"]
/tmp/ipykernel_19096/1095469184.py:13: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  monthly_prices = history.groupby(history.index.to_period("M")).tail(1)["Close"]
/tmp/ipykernel_19096/1095469184.py:13: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  monthly_prices 

,AAPL,SPY,TSLA,BAC,NVDA,GOOG
date,,,,,,
2024-09-30,231.067078,562.210449,261.630005,37.958546,121.250549,166.028030
2024-10-31,224.035904,557.193542,249.850006,40.005707,132.552872,171.489822
2024-11-29,235.620102,590.420898,345.160004,45.448860,138.034332,169.305099
2024-12-31,248.615784,576.215271,403.839996,42.277180,134.089722,189.331009
2025-01-31,234.299667,591.690430,404.600006,44.537727,119.890938,204.402710
2025-02-28,240.361603,584.178955,292.980011,44.345341,124.733696,171.217102
2025-03-31,220.772079,551.628906,259.160004,40.394993,108.228325,155.497162
2025-04-30,211.200958,546.846252,282.160004,38.604176,108.767563,160.135300
2025-05-30,199.883957,581.212769,346.459991,42.718216,134.940887,172.039200


Save the data as a TSV file named "historical_prices.tsv" and include a header row with the column names "date" and the 6 stock ticker symbols.
The date should be the last trading day of the month, so it may not be the last day of the month.
For example, the last trading day of November 2024 was 2024-11-29.
The remaining columns should contain the adjusted closing prices of the corresponding stock tickers on that day.


In [5]:
# YOUR CHANGES HERE

historical_prices.to_csv("historical_prices.tsv", sep="\t")

Submit "historical_prices.tsv" in Gradescope.

## Part 2: Calculate Historical Asset Returns

Calculate the historical asset returns based on the price data that you previously collected.

In [6]:
# YOUR CHANGES HERE

historical_returns = historical_prices.pct_change().dropna()

historical_returns

,AAPL,SPY,TSLA,BAC,NVDA,GOOG
date,,,,,,
2024-10-31,-0.030429,-0.008924,-0.045025,0.053931,0.093215,0.032897
2024-11-29,0.051707,0.059633,0.381469,0.136059,0.041353,-0.012740
2024-12-31,0.055155,-0.024060,0.170008,-0.069786,-0.028577,0.118283
2025-01-31,-0.057583,0.026857,0.001882,0.053470,-0.105890,0.079605
2025-02-28,0.025873,-0.012695,-0.275877,-0.004320,0.040393,-0.162354
2025-03-31,-0.081500,-0.055719,-0.115435,-0.089081,-0.132325,-0.091813
2025-04-30,-0.043353,-0.008670,0.088748,-0.044333,0.004982,0.029828
2025-05-30,-0.053584,0.062845,0.227885,0.106570,0.240635,0.074337
2025-06-30,0.021509,0.051386,-0.083126,0.078605,0.169252,0.027499


Save the data as a TSV file named "historical_returns.tsv" and include a header row with the column names "date" and the 6 stock ticker symbols.
Each row should have the date at the end of the month and the corresponding *relative* price changes.
For example, if the previous price was \$100 and the new price is \$110, the return value should be 0.10.
There should only be 23 rows of data in this file, since they are computed as the differences of 24 prices.

In [7]:
# YOUR CHANGES HERE

historical_returns.index.name = "date"

historical_returns.to_csv(
    "historical_returns.tsv",
    sep="\t"
)

Submit "historical_returns.tsv" in Gradescope.

## Part 3: Estimate Returns

Estimate the expected returns for each asset using the previously calculated return data.
Just compute the average (mean) return for each asset over your data set; do not use other estimators that have been mentioned.
This will serve as your estimate of expected return for each asset.

In [8]:
# YOUR CHANGES HERE
estimated_returns = historical_returns.mean().rename("estimated_return")

estimated_returns

AAPL    0.015696
SPY     0.014246
TSLA    0.027043
BAC     0.023167
NVDA    0.030558
GOOG    0.036176
Name: estimated_return, dtype: float64

Save the estimated returns in a TSV file named "estimated_returns.tsv" and include a header row with the column names "asset" and "estimated_return".

In [10]:
# YOUR CHANGES HERE

estimated_returns.index.name = "asset"

estimated_returns.to_csv(
    "estimated_returns.tsv",
    sep="\t"
)

Submit "estimated_returns.tsv" in Gradescope.

## Part 4: Estimate Risk

Estimate the covariance matrix for the asset returns to understand how the assets move together.

In [11]:
# YOUR CHANGES HERE

estimated_covariance = historical_returns.cov()

estimated_covariance

,AAPL,SPY,TSLA,BAC,NVDA,GOOG
AAPL,0.004014,0.001053,0.002829,0.000360,0.001106,0.002280
SPY,0.001053,0.001380,0.002818,0.001388,0.002279,0.002338
TSLA,0.002829,0.002818,0.026124,0.001914,0.004505,0.005285
BAC,0.000360,0.001388,0.001914,0.004000,0.002539,0.001880
NVDA,0.001106,0.002279,0.004505,0.002539,0.008954,0.002769
GOOG,0.002280,0.002338,0.005285,0.001880,0.002769,0.011466


Save the estimated covariances to a TSV file named "estimated_covariance.tsv".
The header row should have a blank column name followed by the names of the assets.
Each data row should start with the name of an asset for that row, and be followed by the individual covariances corresponding to that row and column's assets.
(This is the format of pandas's `to_csv` method with `sep="\t"` when used on a covariance matrix as computed in the examples.)

In [12]:
# YOUR CHANGES HERE

estimated_covariance.to_csv(
    "estimated_covariance.tsv",
    sep="\t"
)

Submit "estimated_covariance.tsv" in Gradescope.

## Part 5: Construct the Maximum Return Portfolio

Compute the maximum return portfolio based on your previously estimated risks and returns.

In [15]:
# YOUR CHANGES HERE

import cvxpy as cp

n = len(estimated_returns)

x_maximum_return = cp.Variable(n)

objective_maximum_return = cp.Maximize(
    estimated_returns.to_numpy().reshape(1, -1) @ x_maximum_return
)

prob_maximum_return = cp.Problem(
    objective_maximum_return,
    [
        0 <= x_maximum_return,
        cp.sum(x_maximum_return) == 1
    ]
)

estimated_return_maximum_return = prob_maximum_return.solve()

maximum_return_portfolio = pd.Series(
    x_maximum_return.value,
    index=estimated_returns.index,
    name="allocation"
)

maximum_return_portfolio

asset
AAPL    1.944981e-11
SPY     4.405441e-12
TSLA    3.078711e-09
BAC     6.602070e-10
NVDA    1.561501e-08
GOOG    1.000000e+00
Name: allocation, dtype: float64

Save the maximum return portfolio in a TSV file named "maximum_return.tsv".
The header row should have two columns, "asset" and "allocation".
The allocation values should sum up to one.


In [16]:
# YOUR CHANGES HERE

maximum_return_portfolio.index.name = "asset"

maximum_return_portfolio.to_csv(
    "maximum_return.tsv",
    sep="\t"
)

Submit "maximum_return.tsv" in Gradescope.

## Part 6: Construct the Minimum Risk Portfolio

Compute the minimum risk portfolio based on your previously estimated risks.

In [17]:
# YOUR CHANGES HERE

n = len(estimated_returns)

x_minimum_risk = cp.Variable(n)

objective_minimum_risk = cp.Minimize(
    x_minimum_risk.T
    @ estimated_covariance.to_numpy()
    @ x_minimum_risk
)

prob_minimum_risk = cp.Problem(
    objective_minimum_risk,
    [
        0 <= x_minimum_risk,
        cp.sum(x_minimum_risk) == 1
    ]
)

covariance_minimum_risk = prob_minimum_risk.solve()

minimum_risk_portfolio = pd.Series(
    x_minimum_risk.value,
    index=estimated_returns.index,
    name="allocation"
)

minimum_risk_portfolio

asset
AAPL    1.046524e-01
SPY     8.704389e-01
TSLA   -3.707729e-19
BAC     2.490873e-02
NVDA   -3.273524e-19
GOOG   -2.968928e-19
Name: allocation, dtype: float64

Save the minimum risk portfolio in a TSV file named "minimum_risk.tsv".
The header row should have two columns, "asset" and "allocation".
The allocation values should sum up to one.


In [18]:
# YOUR CHANGES HERE

minimum_risk_portfolio.index.name = "asset"

minimum_risk_portfolio.to_csv(
    "minimum_risk.tsv",
    sep="\t"
)

Submit "minimum_risk.tsv" in Gradescope.

## Part 7: Build Efficient Frontier Portfolios

Compute 101 portfolios along the mean-variance efficient frontier with evenly spaced estimated returns.
The first portfolio should be the minimum risk portfolio from part 4, and the last portfolio should be the maximum return portfolio from part 3.
The estimated return of each portfolio should be higher than the previous by one percent of the difference between the first and last portfolios.
That is, the estimated return of the portfolios should be similar to `np.linspace(min_risk_return, max_return, 101)`.


In [21]:
# YOUR CHANGES HERE

import numpy as np

estimated_return_minimum_risk = (
    x_minimum_risk.value.T @ estimated_returns
)

estimated_return_maximum_return = (
    x_maximum_return.value.T @ estimated_returns
)

ef_variances = []
ef_returns = []
ef_portfolios = []

for r in np.linspace(
    estimated_return_minimum_risk,
    estimated_return_maximum_return,
    101
):
    x_r = cp.Variable(n)

    prob_r = cp.Problem(
        cp.Minimize(
            x_r.T @ estimated_covariance.to_numpy() @ x_r
        ),
        [
            0 <= x_r,
            cp.sum(x_r) == 1,
            x_r.T @ estimated_returns == r
        ]
    )

    ef_variances.append(prob_r.solve())
    ef_returns.append(r)
    ef_portfolios.append(x_r.value)

ef_portfolios = np.asarray(ef_portfolios)

Save the portfolios in a TSV file named "efficient_frontier.tsv".
The header row should have columns "index", "return", "risk", and all the asset tickers.
Each data row should have the portfolio index (0-100), the estimated return of the portfolio, the estimated standard deviation (not variance) of the portfolio, and all the asset allocations (which should sum to one).

In [22]:
# YOUR CHANGES HERE

efficient_frontier = pd.DataFrame(
    ef_portfolios,
    columns=estimated_returns.index
)

efficient_frontier.insert(0, "risk", np.sqrt(ef_variances))
efficient_frontier.insert(0, "return", ef_returns)
efficient_frontier.insert(0, "index", range(101))

efficient_frontier

asset,index,return,risk,AAPL,SPY,TSLA,BAC,NVDA,GOOG
0,0,0.014620,0.036683,1.046524e-01,8.704389e-01,2.407936e-20,2.490873e-02,2.021920e-20,4.887157e-20
1,1,0.014836,0.036701,1.122369e-01,8.399228e-01,2.295325e-20,4.784029e-02,1.927397e-20,4.658751e-20
2,2,0.015051,0.036755,1.198214e-01,8.094068e-01,2.182708e-20,7.077185e-02,1.832997e-20,4.430035e-20
3,3,0.015267,0.036844,1.274059e-01,7.788907e-01,2.070147e-20,9.370341e-02,1.738587e-20,4.201520e-20
4,4,0.015482,0.036969,1.349904e-01,7.483747e-01,1.957256e-20,1.166350e-01,1.643727e-20,3.972830e-20
...,...,...,...,...,...,...,...,...,...
96,96,0.035314,0.095638,-2.737935e-22,-3.159426e-22,-9.796173e-23,-1.694445e-22,1.534816e-01,8.465184e-01
97,97,0.035529,0.098289,-3.128458e-22,-3.550554e-22,-1.114281e-22,-1.957615e-22,1.151112e-01,8.848888e-01
98,98,0.035745,0.101087,-4.729905e-22,-5.192853e-22,-1.864150e-22,-2.798226e-22,7.674081e-02,9.232592e-01
99,99,0.035960,0.104020,-4.048287e-22,-4.540984e-22,-1.366262e-22,-2.397222e-22,3.837042e-02,9.616296e-01


In [23]:
efficient_frontier.to_csv(
    "efficient_frontier.tsv",
    sep="\t",
    index=False
)

Submit "efficient_frontier.tsv" in Gradescope.

## Part 8: Check Maximum Return Portfolio Stability

Check the stability of the maximum return portfolio by resampling the estimated risk/return model.

Repeat 1000 times -
1. Use `np.random.multivariate_normal` to generate 23 return samples using your previously estimated risks and returns.
2. Estimate the return of each asset using that resampled return history.
3. Check which asset had the highest return in those resampled estimates.

This procedure is a reduced and simplified version of the Michaud resampled efficient frontier procedure that takes uncertainty in the risk model into account.

In [24]:
# YOUR CHANGES HERE

np.random.seed(704_1)

maximum_return_assets = []

for i in range(1000):

    resampled_returns = np.random.multivariate_normal(
        estimated_returns,
        estimated_covariance,
        23
    )

    resampled_estimated_returns = resampled_returns.mean(axis=0)

    maximum_return_asset = np.argmax(resampled_estimated_returns)

    maximum_return_assets.append(maximum_return_asset)

Save a file "max_return_probabilities.tsv" with the distribution of highest return assets.
The header row should have columns "asset" and "probability".
There should be a data row for each asset and its sample probability of having the highest return based on those 1000 resampled estimates.


In [25]:
# YOUR CHANGES HERE
max_return_probabilities = pd.Series(
    maximum_return_assets
).value_counts(normalize=True).reindex(
    range(len(estimated_returns)),
    fill_value=0
)

max_return_probabilities.index = estimated_returns.index
max_return_probabilities.index.name = "asset"
max_return_probabilities.name = "probability"

max_return_probabilities

asset
AAPL    0.027
SPY     0.000
TSLA    0.285
BAC     0.079
NVDA    0.231
GOOG    0.378
Name: probability, dtype: float64

In [26]:
max_return_probabilities.to_csv(
    "max_return_probabilities.tsv",
    sep="\t"
)

Submit "max_return_probabilities.tsv" in Gradescope.

## Part 9: Acknowledgments

Make a file "acknowledgments.txt" documenting any outside sources or help on this project.
If you discussed this assignment with anyone, please acknowledge them here.
If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for.
If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy.
If no acknowledgments are appropriate, just write none in the file.


Submit "acknowledgments.txt" in Gradescope.

## Part 10: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

Submit "project.ipynb" in Gradescope.